# Journey Console

Run the four cells below in order (Shift+Enter): set up, open the console, follow your runs, stop.

**What the console is for.** Open a policy graph by its path (the top bar; the default is the charge-review graph
built from the PDF; a compact graph from anywhere else becomes a new journey). See it section by section, check the
core segments every graph needs, edit it into a new version and see it on the GMS sphere. See what the graph
decides, edit the **journey factors** and the rest of the test plan with a live check against the graph, and
preview the conversation the graph expects. Then run the test campaign on the device you choose (`cpu`,
`cuda:0`-`cuda:3`) and watch each of its 13 steps report what it produced.

**Needs:** Python 3.12, the knowlytix family installed, and a knowlytix licence in `~/.knowlytix/license.key` (or
`KNOWLYTIX_LICENSE_KEY`). The same setup as `charge_journey_campaign.ipynb`.

In [ ]:
# 1. SETUP: find the repository, the licence and the compute devices
import os, sys
if os.path.exists("../gmsjourney/__init__.py") and not os.path.exists("gmsjourney/__init__.py"):
    os.chdir("..")                                 # Jupyter started in notebooks/: work from the repository root
sys.path.insert(0, os.path.abspath("."))
key = os.path.expanduser("~/.knowlytix/license.key")
if not os.environ.get("KNOWLYTIX_LICENSE_KEY") and os.path.exists(key):
    os.environ["KNOWLYTIX_LICENSE_KEY"] = open(key).read().strip()
os.environ.setdefault("KNOWLYTIX_EULA_ACCEPTED", "1")
from gmsjourney import backend
backend.require()                                  # stops here with a clear message if knowlytix is missing

# Optional: a local LLM as the bot under test (campaign runs only; the Playground uses the gateway).
# The folder of a Llama 3.1 8B Instruct (or any chat model vLLM can serve), with config.json and the weights.
# Leave it empty to use only the model gateway. Set it before opening the console (cell 2).
LOCAL_SUT_MODEL = ""                               # e.g. "/models/Meta-Llama-3.1-8B-Instruct"
if LOCAL_SUT_MODEL:
    os.environ["GMSJOURNEY_SUT_LOCAL_MODEL"] = LOCAL_SUT_MODEL
    os.environ.setdefault("GMSJOURNEY_VLLM_GPU_UTIL", "0.80")    # share of GPU memory vLLM may take
    os.environ.setdefault("GMSJOURNEY_VLLM_MAX_LEN", "16384")    # context length (the bot's prompt is ~6k tokens)

from gmsjourney.app.runs import devices
d = devices()
print(f"repository: {os.getcwd()}")
print(f"torch {d['torch']}: {d['cuda']} CUDA GPU(s) visible" if d["torch"] else "torch not found: CPU only")
for o in d["options"]:
    print(f"  {o['id']:8s} {'ready' if o['available'] else 'not on this machine'}   {o['label']}")
from gmsjourney.sut.local_llm import status as local_status
print("local model for the bot under test:", local_status()["detail"])

## 2. Open the console

The next cell starts the console in the background and shows it here. Use the link it prints to open it in a
**new browser tab**, which gives it more room. Inside:

1. **Graph** (top bar): the path of the policy graph to work on. It starts at
   `gmsjourney/policy/charge_review/charge_review_kg.json`; type another path and press **Open**. A compact graph
   from outside the journey folders becomes a new journey (its customer wording and project are drafted).
2. **Graph Sections** shows the graph split into what the customer uses, what the bot uses, what both use, the
   flow, the rules, the knowledge and the rest, with the **core segments** every graph needs for the checks.
   **Edit Graph** changes it and saves a new version beside the original (`<name>.v2.json`); **GMS Sphere** shows
   every entity on the unit sphere with its relations as arcs. **Overview → Start Here** walks you through the rest.
3. **Journey Factors** is the key part. Each factor is one thing a test case varies in. Press **Preview** on a
   level to see the conversation the graph expects, then **Save Project**.
4. **Playground** (under *Try the bot*): pick a case and a bot and talk to it yourself; click the case's lines or
   type your own. Every bot turn shows the tools it called, whether the graph agrees, and the check of what it said.
5. **Run Campaign**: under **Bot Under Test** choose the journey's reference bot or an **LLM bot** on the model
   gateway (edit the suggested prompt, attach reference files, press *Check Gateway*; the **critique layer**, on by
   default, has a second call review and correct every step the bot proposes), or on a **local model**
   with vLLM when `LOCAL_SUT_MODEL` is set in cell 1 (campaign runs only). Pick the device
   (cuda:0-3 when a GPU is visible), start with 10 to 40 cases, and follow the steps.
6. When a run finishes, step 13 **Verdict** has been written (`out/verdict.html`); **Open the Verdict**: one page that says whether the bot is ready, why, the worst risks and
   how to reproduce the run, then every case with the policy's expected behaviour on the left and the bot's on the
   right. Print it or save it as PDF. **Explore Cases** shows every generated case beside its reworded versions,
   the bot's result on each and the coverage, as pictures.
7. Press **Help** (or `?`) at any time for the glossary.

The page works whether Jupyter runs on your laptop or on a remote GPU machine. Through JupyterHub or
jupyter-server-proxy it is shown via Jupyter; otherwise the cell says which port to forward.

In [ ]:
# 2. OPEN THE CONSOLE (running it again shows the same console)
from gmsjourney import app
app.show(port=8790, height=960)

## 3. Follow your runs from the notebook (optional)

Campaigns started in the console run in their own processes and keep going if you close the tab. This cell lists
them, with their results.

In [ ]:
# 3. RUNS SO FAR
import json, urllib.request
base = next(iter(app.running().values()))
runs = json.load(urllib.request.urlopen(base + "api/runs"))["runs"]
pct = lambda x: "-" if x is None else f"{100 * x:.0f}%"
print(f"{'run':38s} {'status':9s} {'device':7s} {'original':>8s} {'reworded':>8s}  gate (original / reworded)")
for r in runs:
    h = r.get("headline") or {}
    gate = "-" if not h else f"{'pass' if h.get('gate') else 'fail'} / {'-' if h.get('gate_reworded') is None else 'pass' if h['gate_reworded'] else 'fail'}"
    print(f"{r['id']:38s} {r['status']:9s} {r['job'].get('device') or 'auto':7s} {pct(h.get('accuracy_original')):>8s} {pct(h.get('accuracy_reworded')):>8s}  {gate}")
if not runs:
    print("no runs yet: start one from Run Campaign in the console")

## The same campaign from code (optional)

Everything the console does is also a Python call. This cell starts a run only if you set `RUN_FROM_CODE = True`,
so "Run All" never starts a long campaign by accident. It runs in this kernel, so the notebook is busy until it
finishes (a few minutes for 12 cases on a CPU with a cached knowledge store).

In [ ]:
RUN_FROM_CODE = False
if RUN_FROM_CODE:
    from gmsjourney.pipeline import CampaignConfig, run_campaign
    from gmsjourney.domain import set_current
    set_current("charge_review_graph")
    R = run_campaign(CampaignConfig(out_dir="output/notebook_run", device="auto",   # or "cpu", "cuda:0" ... "cuda:3"
                                    max_cases=12, on_step=lambda k, title, R: print(f"step {k}: {title}")))
    print("done:", R["results"]["base_level"]["accuracy"], "original,", R["results"]["enriched"]["accuracy"], "reworded")
else:
    print("skipped: set RUN_FROM_CODE = True to run a campaign from this cell")

## 4. Stop the console

In [ ]:
app.stop()        # stops the page; a campaign already running carries on and shows up next time